#  Apex Predictor — Feature Engineering
Obiettivo: costruire le variabili predittive (forma recente, storico su circuito, affidabilità scuderia, standings) usando solo dati precedenti a ciascuna gara.
Nessuna feature deve poter "vedere" il futuro rispetto alla gara che sta descrivendo. 

In [6]:
# Importiamo le librerie necessarie e scarichiamo il dataset
import pandas as pd 
import numpy as np 

na_vals = ["\\N"]

races = pd.read_csv("../data/raw/races.csv", na_values=na_vals)
results = pd.read_csv("../data/raw/results.csv", na_values=na_vals)
drivers = pd.read_csv("../data/raw/drivers.csv", na_values=na_vals)
constructors = pd.read_csv("../data/raw/constructors.csv", na_values=na_vals)

In [7]:
# Applichiamo la decisione presa in fase di EDA: consideriamo solo le gare dal 2004 in poi
races_recent = races[races["year"] >= 2004].copy()

# Uniamo results con le informazioni di gata che ci servono per l'ordinamento temporale e le feature per circuito
df = results.merge(races_recent[["raceId", "year", "round", "date", "circuitId"]],
                    on="raceId", how="inner") # inner tiene solo le righe con raceId presenti in entrambi i dataframe, quindi solo gare dal 2004 in poi
df["podium"] = df["positionOrder"] <= 3 # creiamo una colonna booleana per indicare se il pilota ha fatto podio o no
print(f"Righe nel dataset di lavoro: {df.shape[0]}")


Righe nel dataset di lavoro: 8650


In [8]:
# 'date' deve essere convertita da stringa a datetime per poter essere ordinata correttamente
df["date"] = pd.to_datetime(df["date"])

# Ordiniamo il dataset per data, poi per pilota
df = df.sort_values(
    ["date", "driverId"]
    ).reset_index(drop=True) # reset_index(drop=True) serve a resettare l'indice del dataframe dopo l'ordinamento
print(df[["date", "driverId", "podium"]].head(10)) # stampiamo le prime 10 righe per verificare l'ordinamento

        date  driverId  podium
0 2004-03-07         2   False
1 2004-03-07         4    True
2 2004-03-07         8   False
3 2004-03-07        11   False
4 2004-03-07        13   False
5 2004-03-07        14   False
6 2004-03-07        15   False
7 2004-03-07        17   False
8 2004-03-07        18   False
9 2004-03-07        21   False


## Feature 1: forma recente del pilota

Media punti e media posizione finale nelle ultime 5 gare del pilota, calcolata escludendo sempre la gara corrente (altrimenti staremmo usando il risultato che vogliamo prevedere come input al modello, un caso di data leakage banale ma facilissimo da introdurre per errore).

In [18]:
N_RACES = 10 # Numero di gare da considerare per la forma recente

df["driver_recent_points_avg"] = (
    df.groupby("driverId")["points"]
    .apply(lambda x: x.shift(1).rolling(N_RACES, min_periods=1).mean())
    .reset_index(level=0, drop=True)  # rimuove l'indice extra che apply()+groupby() aggiunge
)


# 'groupby' applichiamo il calcolo separatamente per ogni pilota
# 'x.shift(1)' sposta i valori di una posizione in avanti così da vedere il valore della gara precedente
# 'rolling(N_RACES, min_periods=1, mean()) calcola la media mobile sulle ultime N gare
# 'min_periods=1' permette di calcolare la media anche quando il pilota ha meno di 5 gare di storico

df["driver_recent_position_avg"] = (
    df.groupby("driverId")["positionOrder"]
    .apply(lambda x: x.shift(1).rolling(N_RACES, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

print(df[["date", "driverId", "points", "driver_recent_points_avg",
          "positionOrder", "driver_recent_position_avg"]].tail(15)) # stampiamo le prime 20 righe per verificare il calcolo della media mobile 

           date  driverId  points  driver_recent_points_avg  positionOrder  \
8635 2024-12-08       825     0.0                     0.900             16   
8636 2024-12-08       830     8.0                    15.000              6   
8637 2024-12-08       832    18.0                    10.200              2   
8638 2024-12-08       840     0.0                     0.000             14   
8639 2024-12-08       842     6.0                     2.800              7   
8640 2024-12-08       844    15.0                    16.400              3   
8641 2024-12-08       846    25.0                    13.900              1   
8642 2024-12-08       847    10.0                    10.600              5   
8643 2024-12-08       848     0.0                     0.800             11   
8644 2024-12-08       852     0.0                     0.800             12   
8645 2024-12-08       855     0.0                     0.400             13   
8646 2024-12-08       857     1.0                    12.700     

In [19]:
# Controlliamo ogni pilota: la sua media ricalcolata manualmente deve coincidere con quella calcolata da 'rolling()'

check_driver = df[df["driverId"] == df["driverId"].iloc[0]].tail(15) # 'iloc' seleziona la prima riga del dataframe, quindi il primo pilota

print(check_driver[["date", "driverId", "points", "driver_recent_points_avg",
                    "positionOrder", "driver_recent_position_avg"]]) # stampiamo le prime 10 righe del pilota selezionato per verificare il calcolo della media mobile

           date  driverId  points  driver_recent_points_avg  positionOrder  \
2575 2010-10-10         2     4.0                       1.3              8   
2599 2010-10-24         2     2.0                       1.7              9   
2623 2010-11-07         2     0.0                       1.9             17   
2647 2010-11-14         2     0.0                       1.9             11   
2671 2011-03-27         2     0.0                       1.5             12   
2695 2011-04-10         2    15.0                       1.3              3   
2719 2011-04-17         2     0.0                       2.8             12   
2743 2011-05-08         2     6.0                       2.5              7   
2767 2011-05-22         2     4.0                       3.1              8   
2791 2011-05-29         2     4.0                       3.1              8   
2815 2011-06-12         2     0.0                       3.5             20   
2839 2011-06-26         2     1.0                       3.1     

## Feature 2: affidabilità della scuderia

Percentuale storica di gare completate (non ritirate) da ciascun costruttore, calcolata con la stessa logica anti-leakage: solo gare precedenti a quella corrente.

In [21]:
# 'position', diversa da positionOrder, è NaN quando il pilota non ha finito la gara, quindi possiamo usarla per calcolare l'affidabilità del costruttore
df["finished"] = df["position"].notnull().astype(int) # creiamo una colonna booleana per indicare se il pilota ha finito la gara o no, 1 = finita, 0 = ritiro

N_RACES_RELIABILITY = 10

df["constructor_reliability"] = (
    df.groupby("constructorId")["finished"]
    .apply(lambda x: x.shift(1).rolling(N_RACES_RELIABILITY, min_periods=1).mean())
    .reset_index(level=0, drop=True)
)

print(df[["date", "constructorId", "finished", "constructor_reliability"]].head(10)) # stampiamo le prime 10 righe per verificare il calcolo della media mobile
print(df[["date", "constructorId", "finished", "constructor_reliability"]].tail(10)) # stampiamo le prime 10 righe per verificare il calcolo della media mobile


        date  constructorId  finished  constructor_reliability
0 2004-03-07             17         0                      NaN
1 2004-03-07              4         1                      NaN
2 2004-03-07              1         0                      NaN
3 2004-03-07             16         1                      NaN
4 2004-03-07             15         0                      NaN
5 2004-03-07              1         1                      0.0
6 2004-03-07              4         1                      1.0
7 2004-03-07             19         0                      NaN
8 2004-03-07             16         1                      1.0
9 2004-03-07             15         1                      0.0
           date  constructorId  finished  constructor_reliability
8640 2024-12-08              6         1                      0.9
8641 2024-12-08              1         1                      1.0
8642 2024-12-08            131         1                      1.0
8643 2024-12-08              3         1   

## Feature 3: storico del pilota sul circuito specifico

Alcuni piloti performano sistematicamente meglio su certi tracciati. Calcoliamo la media posizione finale del pilota su quel circuito specifico, nelle apparizioni precedenti in quel circuito

In [ ]:
# Raggruppiamo per coppia 'driverId' e 'circuitId' per calcolare la media dei punti ottenuti da ogni pilota in ogni circuito

df["driver_circuit_avg_position"] = (
    df.groupby(["driverId", "circuitId"])["positionOrder"]
    .apply(lambda x: x.shift(1).expanding().mean())
    .reset_index(level=[0, 1], drop=True)
    )

# 'expanding()' invece di 'rolling()' perché vogliamo calcolare la media su tutte le gare precedenti in quel circuito, non solo sulle ultime N gare

print(df[["date", "driverId", "circuitId", "positionOrder", "driver_circuit_avg_position"]].tail(20)) # stampiamo le ultime 20 righe per verificare il calcolo della media mobile

           date  driverId  circuitId  positionOrder  \
8630 2024-12-08         1         24              4   
8631 2024-12-08         4         24              9   
8632 2024-12-08       807         24              8   
8633 2024-12-08       815         24             20   
8634 2024-12-08       822         24             18   
8635 2024-12-08       825         24             16   
8636 2024-12-08       830         24              6   
8637 2024-12-08       832         24              2   
8638 2024-12-08       840         24             14   
8639 2024-12-08       842         24              7   
8640 2024-12-08       844         24              3   
8641 2024-12-08       846         24              1   
8642 2024-12-08       847         24              5   
8643 2024-12-08       848         24             11   
8644 2024-12-08       852         24             12   
8645 2024-12-08       855         24             13   
8646 2024-12-08       857         24             10   
8647 2024-

## Analisi feature estratte
Prima di aggiungere nuove feature, esaminiamo quelle che abbiamo ricavato: forma recente del pilota, affidabilità scuderia e storico pilota-circuito

In [23]:
feature_columns = [
    "driver_recent_points_avg", 
    "driver_recent_position_avg", 
    "constructor_reliability", 
    "driver_circuit_avg_position"]

print(df[feature_columns].isnull().sum()) # stampiamo il numero di valori nulli per ogni feature
print("\nPercentuale sul totale:")
print((df[feature_columns].isnull().sum() / len(df) * 100).round(1)) # stampiamo la percentuale di valori nulli per ogni feature

driver_recent_points_avg        108
driver_recent_position_avg      108
constructor_reliability          35
driver_circuit_avg_position    2253
dtype: int64

Percentuale sul totale:
driver_recent_points_avg        1.2
driver_recent_position_avg      1.2
constructor_reliability         0.4
driver_circuit_avg_position    26.0
dtype: float64


In [24]:
# Eliminiamo le righe con NaN nelle feature "strutturali", poiché sono poche e non c'è nessun modo di stimarle

df_clean = df.dropna(subset=[
    "driver_recent_points_avg",
    "driver_recent_position_avg",
    "constructor_reliability",
]).copy() # creiamo una copia del dataframe per evitare il SettingWithCopyWarning

print(f"Righe prima: {df.shape[0]}, righe dopo: {df_clean.shape[0]}, righe eliminate: {df.shape[0] - df_clean.shape[0]}")

# Per la feature 'driver_circuit_avg_position' invece non eliminiamo le righe con NaN, perché sono molte.
# Se il pilota non ha mai corso in quel circuito, usiamo la sua media di posizione recente come stima della sua performance in quel circuito
# Aggiungiamo una colonna booleana per indicare se il pilota ha già corso in quel circuito o no, così da poterla usare come feature nel modello

df_clean["driver_circuit_avg_position"] = df_clean["driver_circuit_avg_position"].fillna(
    df_clean["driver_recent_position_avg"]
) 

# --- Controllo finale: zero NaN residui attesi su tutte e 4 le feature ---
print("\nNaN residui per feature (deve essere tutto 0):")
print(df_clean[[
    "driver_recent_points_avg",
    "driver_recent_position_avg",
    "constructor_reliability",
    "driver_circuit_avg_position"
]].isnull().sum())

Righe prima: 8650, righe dopo: 8519, righe eliminate: 131

NaN residui per feature (deve essere tutto 0):
driver_recent_points_avg       0
driver_recent_position_avg     0
constructor_reliability        0
driver_circuit_avg_position    0
dtype: int64
